# Onboarding: your Academic Cloud API key

Welcome to the Agentic AI lab. Before the first lab, you need one thing: your own **API key** for the Academic Cloud.

**Why?** In the labs you write programs that talk to large language models. Besides the lab server, we use **Chat AI** from the GWDG Academic Cloud: a service that runs large open models and is free for members of Kiel University. Your programs reach it over the internet.

**What is an API key?** A personal password for programs. Your code sends it with every request, so the service knows the request comes from you. It is not your university password.

**Time needed:** about 10 minutes now, then you wait for an approval email. **Request the key today**, because approval is done by hand and can take a while.

This notebook has three steps:

1. Request the key.
2. Save the key on the lab server.
3. Test that it works.

## Step 1: Request the key

1. Open the KISSKI LLM service page: https://kisski.gwdg.de/leistungen/2-02-llm-service
2. Click **Book** (German: *Buchen*).
3. Log in with **Academic Cloud** and choose **Kiel University (CAU)** as your institution. Use your normal CAU login.
4. Under **Service type**, choose **API access to our chat service**.
5. Under **Requirements**, describe what you need. You can copy this text:

   > API access for the Master's lab course "Agentic AI" (infAgAI-01a) at Kiel University. In the weekly labs I write AI agents in Python: agent loops with tool calling, retrieval with embeddings, and multi-agent systems.

6. Submit the form.

The key arrives **by email**, at the address of your Academic Cloud account. It is a long string of letters and digits. Keep the email: you need the key in Step 2.

Waiting for the email? You can already start with **Lab 00**, which runs on the lab server and needs no key.

## Step 2: Save the key on the lab server

Run the cell below. A box appears: paste your key there and press Enter. The key stays hidden while you type.

The key is saved in a private file in your home folder (`~/.academic_cloud_key`) that only you can read. Every lab reads it from there.

**Never paste the key into a notebook cell as plain text.** Notebooks get shared and handed in, and anyone who has your key can use your quota.

In [ ]:
from getpass import getpass
from pathlib import Path

key_file = Path.home() / ".academic_cloud_key"
key = getpass("Paste your Academic Cloud API key and press Enter: ").strip()

if key:
    key_file.write_text(key)
    key_file.chmod(0o600)   # readable by you only
    print(f"Saved to {key_file}")
else:
    print("Nothing entered, nothing saved. Run the cell again.")

## Step 3: Test the key

The cell below sends one real request to the Academic Cloud: it asks a model for a one-word reply. If you see a green check mark, you are ready for the labs.

You do not need to understand this code yet. The labs build it up step by step.

In [ ]:
import json
import urllib.error
import urllib.request
from pathlib import Path

BASE_URL = "https://chat-ai.academiccloud.de/v1"
MODEL = "meta-llama-3.1-8b-instruct"
key_file = Path.home() / ".academic_cloud_key"


def call(path, body=None):
    request = urllib.request.Request(
        BASE_URL + path,
        data=json.dumps(body).encode() if body else None,
        headers={"Authorization": f"Bearer {key_file.read_text().strip()}",
                 "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.load(response)


if not key_file.exists():
    print("No key saved yet. Do Step 2 first.")
else:
    try:
        reply = call("/chat/completions", {
            "model": MODEL,
            "messages": [{"role": "user", "content": "Reply with exactly one word: ready"}],
        })
        print("✅ Your key works. The model replied:", reply["choices"][0]["message"]["content"].strip())
    except urllib.error.HTTPError as error:
        if error.code == 401:
            print("❌ The key was rejected (401). Check that you copied all of it, then redo Step 2.")
        elif error.code == 429:
            print("⏳ Too many requests (429). Wait one minute and run this cell again.")
        else:
            print(f"❌ The server answered with error {error.code}. Try again later, or ask the lab staff.")
    except urllib.error.URLError as error:
        print("❌ Could not reach the Academic Cloud:", error.reason)

## Good to know

- **Your key is personal.** Do not share it, and do not put it in a notebook, a chat, or a git repository.
- **There are limits per key:** about 10 requests per minute and 400 per day. That is enough for the labs. If a cell reports error 429, wait a minute and run it again.
- **Lost the key, or think someone else has it?** Request a new one with the form from Step 1.

## If something goes wrong

| Problem | What to do |
|---|---|
| No email after two working days | Check your spam folder, then write to kisski-support@gwdg.de |
| Error 401 in Step 3 | The key is wrong or incomplete: copy it again from the email and redo Step 2 |
| Error 429 in Step 3 | Too many requests: wait a minute and run the cell again |
| Cannot reach the Academic Cloud | Ask the lab staff; the lab server may have lost its internet connection |

Done? Continue with **Lab 00: LLM Basics**.